In [1]:
pip install nbformat>=4.2.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install yfinance pandas numpy seaborn matplotlib plotly scipy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go
from scipy.ndimage import generic_filter
from sklearn.preprocessing import MinMaxScaler
from scipy.ndimage import zoom

# ==========================================
# 1. CONFIGURATION
# ==========================================
class Config:
    TICKER = "SOL-USD"
    START_DATE = "2020-01-01" 
    END_DATE = "2025-12-01"
    SPLIT_RATIO = 0.70
    
    # Ranges
    FAST_RANGE = range(5, 60, 1)    
    SLOW_RANGE = range(20, 150, 2)
    SIGNAL_RANGE = range(5, 40, 1)
    
    BATCH_SIZE = 2000
    
    # Constraints
    MIN_WIN_RATE = 0.35
    MIN_TRADES = 15

# ==========================================
# 2. DATA & BACKTEST ENGINE
# ==========================================
def fetch_data(ticker, start, end):
    print(f"Fetching data for {ticker}...")
    df = yf.download(ticker, start=start, end=end, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.droplevel(1)
    df['Ret'] = df['Close'].pct_change().fillna(0)
    return df

def precompute_emas(price_series, spans):
    ema_dict = {}
    for span in spans:
        ema_dict[span] = price_series.ewm(span=span, adjust=False).mean()
    return pd.DataFrame(ema_dict)

def calculate_vectorized_metrics(returns_matrix, min_trades, min_win_rate):
    """
    Calculates metrics including Martin Ratio.
    """
    wins = (returns_matrix > 0).sum(axis=0)
    trades = (returns_matrix != 0).sum(axis=0)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        win_rates = np.divide(wins, trades)
        valid_mask = (trades >= min_trades) & (win_rates >= min_win_rate)
    
    if not np.any(valid_mask): return None, valid_mask

    valid_ret = returns_matrix[:, valid_mask]
    cum_returns = (1 + valid_ret).cumprod(axis=0)
    peaks = np.maximum.accumulate(cum_returns, axis=0)
    drawdowns = (cum_returns - peaks) / peaks
    max_dd = drawdowns.min(axis=0)
    
    total_days = returns_matrix.shape[0]
    final_cum = cum_returns[-1, :]
    cagr = (final_cum ** (365 / total_days)) - 1
    ulcer = np.sqrt(np.mean(drawdowns ** 2, axis=0))
    
    with np.errstate(divide='ignore', invalid='ignore'):
        calmar = np.where(max_dd != 0, cagr / np.abs(max_dd), 0)
        sharpe = (valid_ret.mean(axis=0) / valid_ret.std(axis=0)) * np.sqrt(365)
        # FIX: Added Martin Ratio Calculation
        martin = np.where(ulcer != 0, cagr / ulcer, 0)
    
    metrics_df = pd.DataFrame({
        'Calmar': calmar,
        'Ulcer': ulcer,
        'Sharpe': sharpe,
        'Martin': martin, # <--- Added here
        'CAGR': cagr,
        'Win_Rate': win_rates[valid_mask]
    })
    return metrics_df, valid_mask

def perform_batched_grid_search(df):
    print("Preparing Vectorized Backtest...")
    f_list, s_list, sig_list = list(Config.FAST_RANGE), list(Config.SLOW_RANGE), list(Config.SIGNAL_RANGE)
    
    fast_emas = precompute_emas(df['Close'], f_list)
    slow_emas = precompute_emas(df['Close'], s_list)
    
    results = []
    cube_shape = (len(f_list), len(s_list), len(sig_list))
    calmar_cube = np.full(cube_shape, np.nan)
    
    f_map = {v: i for i, v in enumerate(f_list)}
    s_map = {v: i for i, v in enumerate(s_list)}
    sig_map = {v: i for i, v in enumerate(sig_list)}

    valid_pairs = [(f, s) for f in f_list for s in s_list if f < s]
    market_ret = df['Ret'].values[:, None]

    for i in range(0, len(valid_pairs), Config.BATCH_SIZE):
        batch_pairs = valid_pairs[i : i + Config.BATCH_SIZE]
        f_batch_vals = fast_emas[[p[0] for p in batch_pairs]].values
        s_batch_vals = slow_emas[[p[1] for p in batch_pairs]].values
        macd_batch = f_batch_vals - s_batch_vals
        
        for sig in sig_list:
            macd_df = pd.DataFrame(macd_batch)
            sig_line_batch = macd_df.ewm(span=sig, adjust=False).mean().values
            signals = (macd_batch - sig_line_batch) > 0
            positions = np.zeros_like(signals, dtype=float)
            positions[1:] = signals[:-1]
            
            strategy_returns = positions * market_ret
            metrics_df, valid_mask = calculate_vectorized_metrics(
                strategy_returns, Config.MIN_TRADES, Config.MIN_WIN_RATE
            )
            
            if metrics_df is not None:
                current_pairs = np.array(batch_pairs)[valid_mask]
                for idx, (f, s) in enumerate(current_pairs):
                    f_idx, s_idx, sig_idx = f_map[f], s_map[s], sig_map[sig]
                    row_metrics = metrics_df.iloc[idx]
                    
                    calmar_cube[f_idx, s_idx, sig_idx] = row_metrics['Calmar']
                    
                    res = {'F': f, 'S': s, 'Sig': sig, 'Grid_Idx': (f_idx, s_idx, sig_idx)}
                    res.update(row_metrics.to_dict())
                    results.append(res)

    print(f"Grid Search Complete. Strategies Found: {len(results)}")
    return pd.DataFrame(results), calmar_cube

# ==========================================
# 3. STABILITY & RANKING
# ==========================================
def calculate_stability_metrics(calmar_cube, results_df):
    print("Calculating Neighbor Stability (Mean / StDev)...")
    
    def local_mean(buffer): return np.nanmean(buffer)
    def local_std(buffer): return np.nanstd(buffer)

    mean_cube = generic_filter(calmar_cube, local_mean, size=3, mode='constant', cval=np.nan)
    std_cube = generic_filter(calmar_cube, local_std, size=3, mode='constant', cval=np.nan)
    
    with np.errstate(divide='ignore', invalid='ignore'):
        stability_cube = np.divide(mean_cube, std_cube)
        stability_cube[std_cube == 0] = mean_cube[std_cube == 0] * 2 
        
    def get_stability_data(row):
        i, j, k = row['Grid_Idx']
        return pd.Series([mean_cube[i, j, k], std_cube[i, j, k], stability_cube[i, j, k]])

    results_df[['Local_Mean', 'Local_StDev', 'Stability_Ratio']] = results_df.apply(get_stability_data, axis=1)
    return results_df.dropna(subset=['Stability_Ratio']), stability_cube

def rank_strategies(results_df):
    """
    FIX: Updated to use 'Stability_Ratio' and 'Martin'.
    """
    scaler = MinMaxScaler()
    
    # FIX: Check if columns exist before creating copy
    cols_to_use = ['Calmar', 'Martin', 'Stability_Ratio']
    df_norm = results_df[cols_to_use].copy()
    
    df_norm = df_norm.replace([np.inf, -np.inf], np.nan).fillna(0)
    df_norm[:] = scaler.fit_transform(df_norm)
    
    # Weighted Score: 50% Stability, 30% Calmar, 20% Martin
    # Note: We use 'Stability_Ratio' instead of the old 'Robustness'
    results_df['Composite_Score'] = (
        (df_norm['Stability_Ratio'] * 0.50) + 
        (df_norm['Calmar'] * 0.30) + 
        (df_norm['Martin'] * 0.20)
    ) * 100
    
    return results_df.sort_values('Composite_Score', ascending=False)

def rank_strategies_strict(results_df):
    stability_threshold = results_df['Stability_Ratio'].quantile(0.60)
    robust_df = results_df[results_df['Stability_Ratio'] >= stability_threshold].copy()
    return robust_df.sort_values('Calmar', ascending=False)

# ==========================================
# 4. VISUALIZATIONS
# ==========================================
def plot_pareto_frontier(results_df, top_picks):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=results_df['Stability_Ratio'],
        y=results_df['Calmar'],
        mode='markers',
        marker=dict(color=results_df['Composite_Score'], colorscale='Viridis', size=5, opacity=0.6, colorbar=dict(title="Composite Score")),
        name='Strategies'
    ))
    fig.add_trace(go.Scatter(
        x=top_picks['Stability_Ratio'],
        y=top_picks['Calmar'],
        mode='markers',
        marker=dict(color='red', size=12, symbol='star'),
        name='Top Robust'
    ))
    fig.update_layout(title="<b>Pareto Frontier</b>", xaxis_title="Stability Ratio", yaxis_title="Calmar Ratio", template="plotly_dark", height=600)
    fig.show()

def plot_2d_heatmap_slices(results_df):
    """
    Creates a 2D Heatmap of Stability with a slider for Signal Length.
    X-axis: Fast EMA
    Y-axis: Slow EMA
    Color: Stability Ratio
    Slider: Signal Length
    """
    print("Generating 2D Heatmap Slices...")
    
    # Sort for slider consistency
    sig_vals = sorted(results_df['Sig'].unique())
    
    # Create the base figure
    fig = go.Figure()

    # Add a trace for EACH Signal Length (all invisible initially except the first one)
    for sig in sig_vals:
        df_slice = results_df[results_df['Sig'] == sig]
        
        # Pivot the data for heatmap format
        # We use pivot_table to handle potential duplicates (though grid search shouldn't have them)
        pivot = df_slice.pivot_table(index='S', columns='F', values='Stability_Ratio')
        
        fig.add_trace(go.Heatmap(
            z=pivot.values,
            x=pivot.columns,
            y=pivot.index,
            colorscale='Plasma',
            zmin=results_df['Stability_Ratio'].quantile(0.05),
            zmax=results_df['Stability_Ratio'].quantile(0.95),
            colorbar=dict(title='Stability'),
            visible=False,
            name=f"Sig: {sig}"
        ))

    # Make the first trace visible
    fig.data[0].visible = True

    # Create Slider Steps
    steps = []
    for i, sig in enumerate(sig_vals):
        step = dict(
            method="update",
            args=[{"visible": [False] * len(fig.data)},
                  {"title": f"Stability Heatmap (Signal Len: {sig})"}],
            label=str(sig)
        )
        step["args"][0]["visible"][i] = True  # Toggle i-th trace to "True"
        steps.append(step)

    sliders = [dict(
        active=0,
        currentvalue={"prefix": "Signal Length: "},
        pad={"t": 50},
        steps=steps
    )]

    fig.update_layout(
        title=f"<b>Stability Heatmap (Signal Len: {sig_vals[0]})</b>",
        xaxis_title="Fast EMA",
        yaxis_title="Slow EMA",
        sliders=sliders,
        template="plotly_dark",
        height=700,
        width=900
    )
    fig.show()

def plot_3d_scatter_cloud(results_df):
    """
    A cleaner 3D alternative using Scatter points instead of Isosurface.
    Better for seeing the actual data density.
    """
    print("Generating 3D Point Cloud...")
    
    # Filter out the "garbage" to make the chart readable
    # We only show the top 50% of stable strategies to see the "Structure"
    threshold = results_df['Stability_Ratio'].quantile(0.50)
    df_filtered = results_df[results_df['Stability_Ratio'] > threshold]

    fig = go.Figure(data=[go.Scatter3d(
        x=df_filtered['F'],
        y=df_filtered['S'],
        z=df_filtered['Sig'],
        mode='markers',
        marker=dict(
            size=4,
            color=df_filtered['Stability_Ratio'],                # Color by stability
            colorscale='Viridis',   # Viridis is easier to read than Plasma for 3D
            opacity=0.8,
            colorbar=dict(title='Stability Ratio'),
        )
    )])

    fig.update_layout(
        title="<b>3D Stability Cloud (Top 50%)</b><br>Yellow = High Stability, Blue = Medium",
        scene=dict(
            xaxis_title='Fast EMA',
            yaxis_title='Slow EMA',
            zaxis_title='Signal Len',
            xaxis=dict(backgroundcolor="rgb(20, 20, 20)"),
            yaxis=dict(backgroundcolor="rgb(20, 20, 20)"),
            zaxis=dict(backgroundcolor="rgb(20, 20, 20)"),
        ),
        template="plotly_dark",
        margin=dict(l=0, r=0, b=0, t=50),
        height=700
    )
    fig.show()

def plot_method_comparison(ranked_weighted, ranked_strict):
    top_w = ranked_weighted.head(5)
    top_s = ranked_strict.head(5)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=top_w['Stability_Ratio'], y=top_w['Calmar'], mode='markers', marker=dict(size=12, color='cyan'), name='Weighted Picks'))
    fig.add_trace(go.Scatter(x=top_s['Stability_Ratio'], y=top_s['Calmar'], mode='markers', marker=dict(size=12, color='magenta', symbol='x'), name='Strict Picks'))
    fig.update_layout(title="<b>Methodology Face-Off</b>", xaxis_title="Stability", yaxis_title="Calmar", template="plotly_dark", height=500)
    fig.show()

def plot_equity_curves(df, strategies_df, title, split_idx=None):
    fig = go.Figure()
    bnh = df['Close'] / df['Close'].iloc[0]
    fig.add_trace(go.Scatter(x=df.index, y=bnh, mode='lines', name='Buy & Hold', line=dict(color='grey', dash='dash')))

    for i, row in strategies_df.iterrows():
        f, s, sig = int(row['F']), int(row['S']), int(row['Sig'])
        fast = df['Close'].ewm(span=f, adjust=False).mean()
        slow = df['Close'].ewm(span=s, adjust=False).mean()
        macd = fast - slow
        signal = macd.ewm(span=sig, adjust=False).mean()
        pos = np.where((macd - signal).shift(1) > 0, 1, 0)
        pnl = df['Ret'] * pos
        equity = (1 + pnl).cumprod()
        
        # Determine label based on if 'Composite_Score' exists
        if 'Composite_Score' in row:
            label = f"{f}/{s}/{sig} (Sc: {row['Composite_Score']:.1f})"
        else:
            label = f"{f}/{s}/{sig}"
            
        fig.add_trace(go.Scatter(x=df.index, y=equity, mode='lines', name=label))

    if split_idx:
        cutoff_date = df.index[split_idx]
        fig.add_vline(x=cutoff_date, line_dash="dash", line_color="red")
        fig.add_annotation(x=cutoff_date, y=1.02, yref="paper", text="OOS Start", showarrow=False, font=dict(color="red"))
        
    fig.update_layout(title=title, template="plotly_dark", height=600)
    fig.show()

# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    df = fetch_data(Config.TICKER, Config.START_DATE, Config.END_DATE)
    split = int(len(df) * Config.SPLIT_RATIO)
    train_df = df.iloc[:split].copy()
    
    raw_results, calmar_cube = perform_batched_grid_search(train_df)
    
    if raw_results.empty:
        print("No valid strategies found.")
    else:
        rob_results, stability_cube = calculate_stability_metrics(calmar_cube, raw_results)
        
        # Weights: 50% Stability, 30% Calmar, 20% Martin
        ranked_weighted = rank_strategies(rob_results) 
        top_5_weighted = ranked_weighted.head(5)
        
        ranked_strict = rank_strategies_strict(rob_results)
        top_5_strict = ranked_strict.head(5)
        
        print("\n=== METHOD A: COMPOSITE SCORE (Balance) ===")
        print(top_5_weighted[['F', 'S', 'Sig', 'Calmar', 'Stability_Ratio', 'Composite_Score']])
        
        print("\n=== METHOD B: STRICT FILTER (Profit Priority) ===")
        print(top_5_strict[['F', 'S', 'Sig', 'Calmar', 'Stability_Ratio']])
        
        plot_pareto_frontier(ranked_weighted, top_5_weighted)
        plot_method_comparison(ranked_weighted, ranked_strict)
        
        # --- FIX IS HERE ---
        # We pass 'rob_results' (the DataFrame), not 'stability_cube'
        plot_2d_heatmap_slices(rob_results)
        plot_3d_scatter_cloud(rob_results)
        
        comparison_df = pd.concat([
            top_5_weighted.iloc[[0]].assign(Label="Best Weighted"),
            top_5_strict.iloc[[0]].assign(Label="Best Strict")
        ])
        
        plot_equity_curves(df, comparison_df, "Head-to-Head: Balance vs Profit", split)

Fetching data for SOL-USD...
Preparing Vectorized Backtest...


C:\Users\quantico\AppData\Local\Temp\ipykernel_19484\1978936872.py:34: FutureWarning:

YF.download() has changed argument auto_adjust default to True



Grid Search Complete. Strategies Found: 110425
Calculating Neighbor Stability (Mean / StDev)...


C:\Users\quantico\AppData\Local\Temp\ipykernel_19484\1978936872.py:142: RuntimeWarning:

Mean of empty slice

c:\Users\quantico\Desktop\HyperQuery\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning:

Degrees of freedom <= 0 for slice.




=== METHOD A: COMPOSITE SCORE (Balance) ===
         F    S  Sig    Calmar  Stability_Ratio  Composite_Score
110413  59  126   39  3.447243       161.191007        60.734269
91614   52  136   23  6.051618        98.568735        59.454688
91661   53  134   23  6.051618        97.857894        59.229683
91660   53  132   23  6.105523        93.506579        58.279136
92721   51  136   24  6.191199        86.577769        56.663476

=== METHOD B: STRICT FILTER (Profit Priority) ===
        F   S  Sig    Calmar  Stability_Ratio
20659  15  38   15  7.742409        26.002251
18724  16  38   14  7.726799        23.283952
22594  14  38   16  7.726799        23.283952
16789  17  38   13  7.650531        20.459740
24529  13  38   17  7.650531        20.459740


Generating 2D Heatmap Slices...


Generating 3D Point Cloud...
